In [2]:
from dotenv import load_dotenv
load_dotenv()
from getpass import getpass

def deterministic_guardrails(text: str) -> bool:
    """Return true if content if blocked"""
    banned_kw=["hack","exploit","malware","bomb"]
    return any(kw in text.lower() for kw in banned_kw)

text_inputs=[
    'what should i do to avoid hack a database?',
    'what is the capital of France?',
    'explain how to spread a malware?'
]

for inp in text_inputs:
    blocked=deterministic_guardrails(inp)
    status="Blocked" if blocked else "Allow"
    print(f'{status}:{inp}')
    

Blocked:what should i do to avoid hack a database?
Allow:what is the capital of France?
Blocked:explain how to spread a malware?


In [13]:
from langchain_openai import ChatOpenAI

def model_based_gurdrails(text: str) -> str:
    """Use LLM to evaluate content safety, Return SAFETY and UNSAFE"""
    model=ChatOpenAI(model='gpt-4o-mini')
    prompt=f"""Is the following use input safe to process? Reply with only 'SAFE or 'UNSAFE"""

    result=model.invoke([{'role':'user','content':prompt}])
    return result.content.strip()

text=[
    "what should i do to avoid hack a database?",
    "what is the capital of France?",
    "explain how to spread a Malware virus?"
]

for inp in text:
    verdict=model_based_gurdrails(inp)
    status='UNSAFE' if 'UNSAFE' in verdict else "SAFE"
    print(f'{status}:{inp}')

SAFE:what should i do to avoid hack a database?
SAFE:what is the capital of France?
SAFE:explain how to spread a Malware virus?


In [16]:
## BUILD INBUILD GUARDRAILS
#PII personal identifiable information detection middleware

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def customer_lookup(query:str)->str:
    """Look up customer information"""
    return f'Customer record found for query: {query}'

agents=create_agent(model='gpt-3.5-turbo', tools=[customer_lookup],
                    middleware=[PIIMiddleware('email',strategy="redact",apply_to_input=True),
                                PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
                                PIIMiddleware("api_key",detector=r"sk-[a-zA-Z0-9](32)", strategy="block", apply_to_input=True)])

result=agents.invoke({"messages":[{"role":"user","content":"My email is john.doe@example.com and my card is 2345-3233-2424-4422, can you help me"}]})


In [17]:
print(result['messages'][-1].content)

I have found customer records associated with the email [REDACTED_EMAIL] and the card number 2345-3233-2424-4422. How can I assist you further?


In [19]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query:str) -> str:
    """Search the web for information"""
    return f'Search result for :{query}'

@tool
def send_mail(to:str, subject:str, condition:str) -> str:
    """Send an email to a recipient"""
    return f'Email sent to {to} with subject:{subject}'
@tool
def delete_records(table:str, condition: str) -> str:
    """delete record from database"""
    return f'Deleted records from {table} where {condition}'

agent=create_agent(model='gpt-3.5-turbo', tools=[search_web,send_mail,delete_records],
                   middleware=[HumanInTheLoopMiddleware(interrupt_on={'search_web':True,
                                                                      'delete_records':True,
                                                                      'send_mail':True})],
                                        checkpointer=InMemorySaver())
config = {'configurable':{"thread_id":"1"}}
result=agent.invoke({'messages':[{'role':'user',"content":'search a email to team@company.com about the Q4 result'}]},config)
print(result)

{'messages': [HumanMessage(content='search a email to team@company.com about the Q4 result', additional_kwargs={}, response_metadata={}, id='341c4d18-3448-4099-ba77-72eeb4540fbe'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 111, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EbrlxEcgq7uNXgRMW4k7vYoaVIwrF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a164dc-9fb7-7790-bf90-ad7b6f39f82e-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'email to:team@company.com Q4 result'}, 'id': 'call_ZK7Ow1tNnaHix0Xxna1kFxqL', 'type': 'tool_call'}]

In [20]:
approved_result= agent.invoke(Command(resume={"decisions":[{'type':"approve"}]}),config)
print(approved_result['messages'][-1].content)

I couldn't find any specific email with that exact query in the search results. Would you like me to try searching again or do something else?
